In [9]:
%pip install torch torchvision timm ipywidgets tqdm

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import os
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# 1. Tentukan Path (Sesuai struktur folder yang kita temukan tadi)
base_dir = "./DATASET/real_vs_fake/real-vs-fake"

# 2. Transformasi Gambar
# MobileNet butuh ukuran 224x224. Kita tambahkan Augmentasi untuk data Train agar AI lebih pintar
data_transforms = {
    'train': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.RandomHorizontalFlip(), # Biar AI gak cuma kenal muka hadap satu sisi
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]) # Standar ImageNet
    ]),
    'valid': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
}

# 3. Load Dataset menggunakan ImageFolder
image_datasets = {
    'train': datasets.ImageFolder(os.path.join(base_dir, 'train'), data_transforms['train']),
    'valid': datasets.ImageFolder(os.path.join(base_dir, 'valid'), data_transforms['valid'])
}

# 4. DataLoader (Pengatur antrean gambar ke GPU/CPU)
# Batch size 32-64 biasanya aman untuk VRAM 8GB-12GB
dataloaders = {
    'train': DataLoader(image_datasets['train'], batch_size=32, shuffle=True, num_workers=2),
    'valid': DataLoader(image_datasets['valid'], batch_size=32, shuffle=False, num_workers=2)
}

# Cek hasil
dataset_sizes = {x: len(image_datasets[x]) for x in ['train', 'valid']}
class_names = image_datasets['train'].classes

print(f"✅ Data Loader Berhasil Dibuat!")
print(f"Total Gambar Latih: {dataset_sizes['train']:,}")
print(f"Kelas yang ditemukan: {class_names}")

✅ Data Loader Berhasil Dibuat!
Total Gambar Latih: 100,000
Kelas yang ditemukan: ['fake', 'real']


In [ ]:
import timm
import torch
import torch.nn as nn

# 1. Memanggil MobileNetV4 
# 'mobilenetv4_conv_medium' adalah pilihan seimbang antara speed dan akurasi
model = timm.create_model('mobilenetv4_conv_medium', pretrained=True, num_classes=2)

# 2. Cek apakah GPU tersedia
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

print(f"✅ MobileNetV4 Berhasil Dimuat!")
print(f"Perangkat aktif: {device}")

# Opsional: Cek struktur layer terakhir
print(f"Output layer: {model.get_classifier()}")

✅ MobileNetV4 Berhasil Dimuat!
Perangkat aktif: cuda
Output layer: Linear(in_features=1280, out_features=2, bias=True)


In [ ]:
import torch.optim as optim

# Loss function
criterion = nn.CrossEntropyLoss()

# Optimizer AdamW (Weight Decay membantu generalisasi model)
optimizer = optim.AdamW(model.parameters(), lr=0.0001)

print("Loss function: CrossEntropyLoss")
print("Optimizer: AdamW")

Loss function: CrossEntropyLoss
Optimizer: AdamW


In [22]:
from tqdm import tqdm 
import time
import sys

def train_model(model, criterion, optimizer, num_epochs=5):
    since = time.time()
    best_acc = 0.0
    
    if not os.path.exists('checkpoints'):
        os.makedirs('checkpoints')

    for epoch in range(num_epochs):
        print(f'\nEpoch {epoch+1}/{num_epochs}')
        print('-' * 10)

        for phase in ['train', 'valid']:
            if phase == 'train':
                model.train()
            else:
                model.eval()

            running_loss = 0.0
            running_corrects = 0

            pbar = tqdm(dataloaders[phase], 
            desc=f"{phase.upper()}", 
            file=sys.stdout, # Paksa output ke layar utama
            dynamic_ncols=True)
            
            for inputs, labels in pbar:
                inputs = inputs.to(device)
                labels = labels.to(device)
                optimizer.zero_grad()

                with torch.set_grad_enabled(phase == 'train'):
                    outputs = model(inputs)
                    _, preds = torch.max(outputs, 1)
                    loss = criterion(outputs, labels)
                    if phase == 'train':
                        loss.backward()
                        optimizer.step()

                running_loss += loss.item() * inputs.size(0)
                running_corrects += torch.sum(preds == labels.data)
                
                # Update info di samping bar progres
                pbar.set_postfix({'loss': loss.item()})

            epoch_loss = running_loss / dataset_sizes[phase]
            epoch_acc = running_corrects.double() / dataset_sizes[phase]

            print(f'{phase.upper()} Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}')

            if phase == 'valid':
                checkpoint_path = f'checkpoints/mobilenetv4_epoch_{epoch+1}.pth'
                torch.save({
                    'epoch': epoch + 1,
                    'model_state_dict': model.state_dict(),
                    'optimizer_state_dict': optimizer.state_dict(),
                    'loss': epoch_loss,
                }, checkpoint_path)

                if epoch_acc > best_acc:
                    best_acc = epoch_acc
                    torch.save(model.state_dict(), 'checkpoints/best_mobilenetv4_model.pth')
                    print("🌟 Model Terbaik Disimpan!")

    time_elapsed = time.time() - since
    print(f'\nTraining selesai dalam {time_elapsed // 60:.0f}m {time_elapsed % 60:.0f}s')
    return model

# Jalankan dengan 5 Epoch dulu (Sesuaikan jika ingin lebih lama)
model_ft = train_model(model, criterion, optimizer, num_epochs=5)


Epoch 1/5
----------
TRAIN: 100%|██████████| 3125/3125 [08:12<00:00,  6.34it/s, loss=0.254]  
TRAIN Loss: 0.1391 Acc: 0.9436
VALID: 100%|██████████| 625/625 [01:20<00:00,  7.80it/s, loss=0.0951] 
VALID Loss: 0.2940 Acc: 0.9428
🌟 Model Terbaik Disimpan!

Epoch 2/5
----------
TRAIN: 100%|██████████| 3125/3125 [08:05<00:00,  6.44it/s, loss=0.111]   
TRAIN Loss: 0.0944 Acc: 0.9641
VALID: 100%|██████████| 625/625 [00:36<00:00, 17.31it/s, loss=0.0571]
VALID Loss: 0.1963 Acc: 0.9348

Epoch 3/5
----------
TRAIN: 100%|██████████| 3125/3125 [07:57<00:00,  6.55it/s, loss=0.159]   
TRAIN Loss: 0.0587 Acc: 0.9786
VALID: 100%|██████████| 625/625 [01:24<00:00,  7.43it/s, loss=0.011]   
VALID Loss: 0.0422 Acc: 0.9858
🌟 Model Terbaik Disimpan!

Epoch 4/5
----------
TRAIN: 100%|██████████| 3125/3125 [07:54<00:00,  6.59it/s, loss=0.00969] 
TRAIN Loss: 0.0309 Acc: 0.9888
VALID: 100%|██████████| 625/625 [00:35<00:00, 17.69it/s, loss=0.0278]  
VALID Loss: 0.0453 Acc: 0.9858

Epoch 5/5
----------
TRAIN: 100